In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 64
batch_size = 128
max_iters = 1000
learning_rate = 3e-3
eval_iters = 100
n_embd = 384
n_head = 4
n_layer = 4
#20% of the neurons will dropout to prevent overfitting
dropout = 0.2

In [17]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [18]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [20]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x , y = x.to(device), y.to(device)
    return x, y

In [21]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x , y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
            

In [ ]:
class Head(nn.Module):
    #One Head of Self-Attention
    #Scaled Dot Product Attention Layer

    def __init__(self, head_size):
        super().__init__()
        #Inputs of K,Q,V
        #Key
        self.key = nn.Linear(n_embd, head_size, bias=False)
        #Query
        self.query = nn.Linear(n,embd, head_size, bias=False)
        #Value
        self.value = nn.Linear(n_embd, head_size, bias=False)
        #Register Buffer
        #Registers No LookAhead Masking in the Model's State
        #That way we dont have to re initialize it every time
        #Saves computation time
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))
        #Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        #Input of size(batch, time-step, channels)
        #Output of size(batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x) #(B,T,head size)
        q = self.query(x) #(B,T, head size)
        #Compute Attention Scores or 'affinities'
        #wei = weights
        # (B,T,had size) @ (B,head size, T) -> (B,T,T)
        # Dot product first and scaled by 1/sqrt(dk)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        #torch.tril and masking for no look ahead
        # every value at 0 gets changed to -inf for softmax
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #(B,T,T)
        #Softmax Activation Function is used
        #Converts logits (raw, unscaled numbers) into positive decimals that represent percentages
        wei = F.softmax(wei, dim=-1 #(B,T,T)
        #weights get dropout
        wei = self.dropout(wei)
        #Perform the weighted aggregation of the values
        v = self.value(x)
        #Matrix multiplication
        # (B,T,T) @ (B,T, head size) -> (B,T, head size)
        out = wei @ v
        return out
        
        
class MultiHeadAttention(nn.Module):
    #Multi-Head Attention Layer
    #Multiple Heads of Self-Attention in parallel

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        #(B,T,F) -> (B,T,(h1,h1,h1,h1,h2,h2,h2,h2,h3,h3,h3,h3))
        #Concatenate data above so it's easier to process
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
    


class FeedForward(nn.Module):
    #Feed Forward Layer
    #Linear layer followed by ReLu Activation Function and final Linear Transformation
    #Layer Initialization
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            #Linear Transformation of the data
            nn.Linear(n_embed, 4 * n_embd),
            #ReLu Non-Linear Activation Function for Neurons
            nn.ReLU(),
            #Final Linear Transformation of the data
            nn.Linear(4 * n_embd, n_embd),
            #Dropout is applied after the Activation Function
            #Dropout makes a certain percentage of neurons to "dropout" or become 0
            #Dropout is designed to prevent the co adaptation of neurons
            #In other words prevents overfitting*
            #Or *the adaptability of a system that then performs poorly on new data
            nn.Dropout(dropout),
        )

    #Forward Pass
    def forward(self, x):
        return self.net(x)


class Decoder_Block(nn.Module):
    #Transformer Block

    #Initialization Function for Layer
    def __init__(self, n_embd, n_head):
        super().__init__()

        #Initialize head size
        head_size = n_embd // n_head
        
        #Self Attention Block
        self.sa = MultiHeadAttention(n_head, head_size)

        #Feed Forward Pass
        self.ffwd = FeedForward(n_embd)

        #Layer Normalization
        self.ln1 = nn.LayerNorm(n_embd)

        #Layer Normalization
        self.ln2 = nn.LayerNorm(n_embd)

    #Actual Layer Progression
    #Forward Pass just means feeding the data through the distinct layers
    def forward(self, x):
        #Block starts off with Self Attention Block
        y = self.sa(x)
        #Passes to Residual Connection (Addition) and Normalization
        x = self.ln1(x + y)
        #Passes to Feed Forward Layer
        y = self.ffwd(x)
        #Passes to 2nd Residual Connection (Addition) and Normalization Layer
        x = self.ln2(x + y)
        #Finishes Block
        return x
    

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        #Token Embedding Table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        #Positional Embedding Table
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        #Setting Up Decoder Layers
        self.blocks = nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layer)])
        #Final Layer Normalization
        self.ln_f = nn.LayerNorm(n_embd)
        #Final Linear NN after Decoder Blocks
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape

        #Index and targets are both (B,T) tensor of integers
        tok_emb =self.token_embedding_table(index) #(B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        x = self.blocks(x) #(B,T,C)
        x = self.ln_f(x) #(B,T,C)
        logits = self.lm_head(x) #(B,T, vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.coss_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        #Index is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #Get the predictions
            logits, loss = self.forward(index)
            #Focus only on the last time step
            logits = logits[:,-1,:] #Becomes (B,C)
            #Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) #(B,C)
            #Sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) #(B,1)
            #Append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=-1)) #(B, T+1)
        return index
model = GPTLanguageModel(vocab_size)
m = model.to(device)

        
        
        
        

In [ ]:
#Create PyTorch Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}')

    #Sample a batch of data
    xb, yb = get_batch('train')

    #Evaluate the loss
    logits, loss = model.forward(xb,yb)
    #Set gradient to None instead of 0 because None occupies less space
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())